In [2]:
pip install --upgrade pyBKT

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install scikit-learn==1.3.0

Note: you may need to restart the kernel to use updated packages.


In [4]:
import random as rand
import matplotlib.pyplot as plt
import itertools
import pandas as pd
from pyBKT.models import Model
import os
import joblib
import json
from ELO import Elo

In [5]:
import numpy as np

In [6]:
with open('../elo_variable.json', 'r') as Elo_Data:
    elo_data = json.load(Elo_Data)

print(elo_data)

{'globals': {'students': 500, 'init_skill_level': [0.0, 0.0], 'k_success': 1, 'k_fail': 0.5}, 'scenarios': [{'id': 1, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 0], [0, 1]], 'difficulty_level': [1, 1], 'depends': [-1, -1]}, {'id': 2, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 1], [1, 1]], 'difficulty_level': [0, 1], 'depends': [-1, -1]}, {'id': 3, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0], [0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1], 'depends': [-1, -1, -1, -1]}, {'id': 4, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1], [0, 1, 1, 1]], 'difficulty_level': [1, 1, 0, 1], 'depends': [-1, -1, -1, -1]}, {'id': 5, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1, 0, 1], [0, 1, 1, 0, 1, 1]], 'difficulty_level': [0, 0, 0, 1, 1, 1], 'depends': [-1, -1, -1, -1, -1, -1]}, {'id': 6, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1, 2, 2], 'depends': [-1, -1, -1, -1, -1, -1]}

In [7]:
def load_training_data(scenario, split):
    base_path = "../Simulated_Data"
    
    filename = f"scenario_{scenario}_{split}_data.csv"
    filepath = os.path.join(base_path, filename)
    
    return pd.read_csv(filepath)

In [8]:
import joblib

def save_bkt(model, scenario_id, num_Skills, base_dir=None):
    # Load the tarin model's parameters
    model_params     = model.params() #
    model_parameters = {} # Initialize the dictionary to store the parameters for each skill
    
    for skill_id in range(num_Skills):
         # Get the parameters for the current skill
        params = model_params.loc[(str(skill_id),), :]
        
        p_start = params.loc[('prior', 'default'), 'value']
        p_trans = params.loc[('learns', 'default'), 'value']
        
        guesses = params.loc[params.index.get_level_values('param').isin(['guesses'])].reset_index(level=0, drop=True)
        p_guesses = guesses.reset_index(level=0)[['class', 'value']]
        
        # Extract slips for all classes for the current skill
        slips = params.loc[params.index.get_level_values('param').isin(['slips'])].reset_index(level=0, drop=True)
        p_slips = slips.reset_index(level=0)[['class', 'value']]
        
        # Store the extracted parameters in the dictionary for future use
        model_parameters[f"p_start_{skill_id}"] = p_start
        model_parameters[f"p_trans_{skill_id}"] = p_trans
        model_parameters[f"p_guesses_{skill_id}"] = p_guesses.to_dict(orient='records')
        model_parameters[f"p_slips_{skill_id}"] = p_slips.to_dict(orient='records')
    
    if base_dir is None:
        base_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'Trained_Models'))

    os.makedirs(base_dir, exist_ok=True)

    model_path = os.path.join(base_dir, f'BKT_model_scenario_{scenario_id}.pth')
    
    # Save the model parameters to a JSON file
    with open(model_path, 'w') as f:
        json.dump(model_parameters, f, indent=4)
    
    # Name the model
    model_filename = f'BKT_model_scenario_{scenario_id}.pkl'    # Filename
    
    # Save the model to the specified path using joblib
    joblib.dump(model, os.path.join(base_dir, model_filename))
     

In [9]:
all_data = {}

level_skill   = [] 
mastery_level = 1.5
i = 0

global_values = elo_data["globals"]
scenarios = elo_data["scenarios"]

students          = global_values["students"]
init_skill_level  = np.array(global_values["init_skill_level"])
k_success         = global_values["k_success"]

In [10]:
defaults = {
    'user_id' :   'student_id',
    'order_id':   'step',           # Assuming 'Task_ID' corresponds to the order ID
    'skill_name': 'skill_id',     # Assuming 'Skill' corresponds to the skill ids
    'correct':    'success',        # Assuming 'Success' corresponds to correct/incorrect values
    'multigs': 'task'
    
}

In [13]:
for scenario in scenarios:
    scenario_id    = scenario["id"]
    num_tasks      = scenario["num_tasks"]
    difficulty_level = np.array(scenario["difficulty_level"])
    q_matrix       = np.array(scenario["q_matrix"])
    num_skills = scenario["num_skills"]
    train_df = load_training_data(scenario_id, "train")
    test_df = load_training_data(scenario_id, "test")

    model = Model(seed = 42, num_fits = 1)
    model.fit(data=train_df, defaults=defaults, multigs = True)#, forgets = True)
    # Perform evalution
    training_acc = model.evaluate(data=train_df, metric='accuracy')
    print(f"Scenario {scenario_id} train Accuracy, {training_acc}")
    print(model.params())
    test_acc = model.evaluate(data=test_df, metric='accuracy')
    print(f"Scenario {scenario_id} test Accuracy, {test_acc}")

    save_bkt(model, scenario_id, num_skills)

Scenario 1 train Accuracy, 0.6302142051860203
                        value
skill param   class          
1     prior   default 0.03917
      learns  default 0.25404
      guesses 1       0.25419
      slips   1       0.23160
      forgets default 0.00000
0     prior   default 0.00421
      learns  default 0.30127
      guesses 0       0.27395
      slips   0       0.27205
      forgets default 0.00000
Scenario 1 test Accuracy, 0.6485436893203883


/var/folders/fg/s74_sj917k33_5w_bs1z5_gr0000gn/T/ipykernel_42891/2836936374.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  params = model_params.loc[(str(skill_id),), :]
/var/folders/fg/s74_sj917k33_5w_bs1z5_gr0000gn/T/ipykernel_42891/2836936374.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  params = model_params.loc[(str(skill_id),), :]


Scenario 2 train Accuracy, 0.7043269230769231
                        value
skill param   class          
0     prior   default 0.03142
      learns  default 0.36772
      guesses 0       0.22576
              1       0.05908
      slips   0       0.33296
              1       0.67688
      forgets default 0.00000
1     prior   default 0.01348
      learns  default 0.35588
      guesses 0       0.23524
              1       0.06500
      slips   0       0.32581
              1       0.67224
      forgets default 0.00000
Scenario 2 test Accuracy, 0.7317073170731707
Scenario 3 train Accuracy, 0.676923076923077
                        value
skill param   class          
1     prior   default 0.07099
      learns  default 0.26871
      guesses 1       0.51556
              3       0.24299
      slips   1       0.08097
              3       0.25676
      forgets default 0.00000
0     prior   default 0.09356
      learns  default 0.34126
      guesses 0       0.44236
              2       0.

/var/folders/fg/s74_sj917k33_5w_bs1z5_gr0000gn/T/ipykernel_42891/2836936374.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  params = model_params.loc[(str(skill_id),), :]
/var/folders/fg/s74_sj917k33_5w_bs1z5_gr0000gn/T/ipykernel_42891/2836936374.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  params = model_params.loc[(str(skill_id),), :]


Scenario 4 train Accuracy, 0.6722776392352452
                        value
skill param   class          
0     prior   default 0.02991
      learns  default 0.36188
      guesses 0       0.27816
              2       0.21789
              3       0.04991
      slips   0       0.33296
              2       0.33072
              3       0.66409
      forgets default 0.00000
1     prior   default 0.00237
      learns  default 0.36120
      guesses 1       0.26180
              2       0.22332
              3       0.07038
      slips   1       0.36398
              2       0.33867
              3       0.66957
      forgets default 0.00000
Scenario 4 test Accuracy, 0.6722129783693843
Scenario 5 train Accuracy, 0.6867421180274859
                        value
skill param   class          
0     prior   default 0.00377
      learns  default 0.40431
      guesses 0       0.49617
              2       0.23361
              3       0.24013
              5       0.07048
      slips   0       0

/var/folders/fg/s74_sj917k33_5w_bs1z5_gr0000gn/T/ipykernel_42891/2836936374.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  params = model_params.loc[(str(skill_id),), :]
/var/folders/fg/s74_sj917k33_5w_bs1z5_gr0000gn/T/ipykernel_42891/2836936374.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  params = model_params.loc[(str(skill_id),), :]
/var/folders/fg/s74_sj917k33_5w_bs1z5_gr0000gn/T/ipykernel_42891/2836936374.py:10: PerformanceWarning: indexing past lexsort depth may impact performance.
  params = model_params.loc[(str(skill_id),), :]


Scenario 8 train Accuracy, 0.7311178247734139
                        value
skill param   class          
0     prior   default 0.04181
      learns  default 0.27769
      guesses 0       0.54658
              2       0.22361
              4       0.07969
              6       0.01288
      slips   0       0.08517
              2       0.21262
              4       0.47522
              6       0.73128
      forgets default 0.00000
1     prior   default 0.01988
      learns  default 0.22438
      guesses 1       0.53650
              3       0.31746
              5       0.10995
              7       0.01858
      slips   1       0.01367
              3       0.18348
              5       0.39496
              7       0.59836
      forgets default 0.00000
Scenario 8 test Accuracy, 0.7768456375838926
